# 🔍 Exercice 1 — Embeddings & Recherche Sémantique

**Durée estimée : 35 minutes**

## Objectif

Comprendre comment transformer du texte en vecteurs numériques (embeddings)
et utiliser la **similarité cosinus** pour retrouver des films pertinents
à partir d'une requête en langage naturel.

## Ce que vous allez apprendre

1. Ce qu'est un **embedding** et pourquoi c'est utile
2. Comment encoder du texte avec **sentence-transformers**
3. Comment mesurer la proximité sémantique entre deux textes
4. Comment construire un **moteur de recherche sémantique** simple

## Prérequis

- Python 3.10+
- Les dépendances du projet (voir README pour l'installation avec `uv`)

> **Analogie pour les habitués de l'IA prédictive** : un modèle d'embedding
> fait du *feature engineering automatique* sur du texte. Au lieu d'encoder
> manuellement vos variables catégorielles, le modèle transforme une phrase
> entière en un vecteur dense de 384 dimensions, prêt à être utilisé
> pour des calculs de distance.

---
## Partie 1 — Chargement des données

On commence par charger notre dataset de films d'horreur (généré via l'API TMDB).

In [1]:
import json
import numpy as np
from pathlib import Path

# Charger le dataset
dataset_path = Path("horror_movies_dataset.json")
with open(dataset_path, encoding="utf-8") as f:
    movies = json.load(f)

print(f"Nombre de films chargés : {len(movies)}")
print(f"\nExemple — premier film :")
for key, value in movies[0].items():
    print(f"  {key}: {value}")

Nombre de films chargés : 50

Exemple — premier film :
  id: 346364
  title: Ça
  original_title: It
  overview: À Derry, dans le Maine, aux États-Unis, 7 gamins, ayant du mal à s’intégrer, se sont regroupés au sein du Club des Ratés. Rejetés par leurs camarades, ils sont les cibles favorites des gros durs de l’école. Ils ont aussi en commun d’avoir éprouvé leur plus grande terreur face à un terrible prédateur métamorphe qu’ils appellent Ça…
  release_date: 2017-09-06
  vote_average: 7.239
  vote_count: 20073
  popularity: 16.6551


Chaque film contient un `title`, un `overview` (synopsis), une `release_date`
et un `vote_average`. C'est le champ **overview** qui nous intéresse :
c'est le texte qu'on va transformer en vecteur.

---
## Partie 2 — Qu'est-ce qu'un embedding ?

Un **embedding** est une représentation vectorielle d'un texte dans un espace
à N dimensions. Le modèle a été entraîné pour que :

- Deux textes **proches en sens** → vecteurs **proches** dans l'espace
- Deux textes **éloignés en sens** → vecteurs **éloignés** dans l'espace

Et cela fonctionne **même sans mots en commun** :
- *"film de zombies effrayant"* sera proche de *"morts-vivants terrorisent une ville"*
- *"recette de crêpes"* sera très éloigné des deux

On utilise le modèle **paraphrase-multilingual-MiniLM-L12-v2** de la bibliothèque
`sentence-transformers`. Il est **multilingue** (50+ langues dont le français),
rapide sur CPU, et produit des vecteurs de **384 dimensions**.

In [2]:
from sentence_transformers import SentenceTransformer

# Charger le modèle d'embedding (téléchargement ~470 Mo la première fois)
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print(f"Modèle chargé : paraphrase-multilingual-MiniLM-L12-v2")
print(f"Dimension des vecteurs : {model.get_sentence_embedding_dimension()}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

Modèle chargé : paraphrase-multilingual-MiniLM-L12-v2
Dimension des vecteurs : 384


---
## Partie 3 — Encoder du texte en vecteurs

Transformons quelques phrases en vecteurs pour observer le résultat.

In [3]:
# Encoder une phrase simple
phrase = "Un film d'horreur avec des fantômes dans un manoir"
vecteur = model.encode(phrase)

print(f"Phrase : '{phrase}'")
print(f"Type   : {type(vecteur)}")
print(f"Shape  : {vecteur.shape}")
print(f"\nPremières 10 valeurs du vecteur :")
print(np.round(vecteur[:10], 4))

Phrase : 'Un film d'horreur avec des fantômes dans un manoir'
Type   : <class 'numpy.ndarray'>
Shape  : (384,)

Premières 10 valeurs du vecteur :
[-0.0389 -0.0266 -0.2625  0.2228  0.2098 -0.0332  0.0414 -0.1805  0.3795
 -0.0589]


Chaque phrase est maintenant un tableau de **384 nombres décimaux**.
Ces nombres n'ont pas de signification individuelle — c'est leur combinaison
qui capture le sens de la phrase.

Encodons maintenant les synopsis de tous nos films :

In [4]:
# Encoder tous les synopsis des films
overviews = [m["overview"] for m in movies]
embeddings = model.encode(overviews, show_progress_bar=True)

print(f"Matrice d'embeddings : {embeddings.shape}")
print(f"  → {embeddings.shape[0]} films × {embeddings.shape[1]} dimensions")

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Matrice d'embeddings : (50, 384)
  → 50 films × 384 dimensions


---
## Partie 4 — Similarité cosinus

Pour mesurer la proximité entre deux vecteurs, on utilise la **similarité cosinus**.

Elle mesure l'angle entre deux vecteurs :
- **1.0** = même direction (textes très similaires)
- **0.0** = perpendiculaires (aucun lien)
- **-1.0** = directions opposées (rare en pratique avec des embeddings)

### Implémentation manuelle

La formule est simple :

$$\text{cosine\_sim}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}$$

Implémentons-la à la main pour bien comprendre :

In [5]:
def cosine_similarity_manual(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    """Calcule la similarité cosinus entre deux vecteurs."""
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    return dot_product / (norm_a * norm_b)


# Test : comparer deux phrases proches et deux phrases éloignées
phrases = [
    "Un film de zombies terrifiant",
    "Des morts-vivants attaquent une ville",
    "Recette de gâteau au chocolat",
]
vecs = model.encode(phrases)

print("Similarités cosinus :\n")
print(f"  Zombies vs Morts-vivants : {cosine_similarity_manual(vecs[0], vecs[1]):.4f}")
print(f"  Zombies vs Gâteau       : {cosine_similarity_manual(vecs[0], vecs[2]):.4f}")
print(f"  Morts-vivants vs Gâteau : {cosine_similarity_manual(vecs[1], vecs[2]):.4f}")

Similarités cosinus :

  Zombies vs Morts-vivants : 0.5253
  Zombies vs Gâteau       : 0.1652
  Morts-vivants vs Gâteau : 0.0238


Les deux premières phrases parlent du même sujet (zombies/morts-vivants)
→ score élevé (~0.5-0.7), alors qu'elles n'ont **aucun mot en commun**.

La phrase sur le gâteau est très éloignée des deux autres → score faible (~0.1-0.3).

C'est la puissance de la recherche **sémantique** par rapport à une recherche par mots-clés.

> **Note** : le modèle `paraphrase-multilingual-MiniLM-L12-v2` est multilingue
> et produit des scores bien contrastés sur du texte français. Avec un modèle
> exclusivement anglophone, les écarts seraient plus faibles.

### Avec scikit-learn

En production, on utilise `sklearn` pour calculer efficacement la similarité
sur des matrices entières :

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

# Vérification : même résultat que notre implémentation manuelle
sim_sklearn = cosine_similarity([vecs[0]], [vecs[1]])[0][0]
sim_manual = cosine_similarity_manual(vecs[0], vecs[1])

print(f"sklearn : {sim_sklearn:.6f}")
print(f"manuel  : {sim_manual:.6f}")
print(f"Écart   : {abs(sim_sklearn - sim_manual):.10f}")

sklearn : 0.525320
manuel  : 0.525320
Écart   : 0.0000001192


---
## Partie 5 — Moteur de recherche sémantique

Assemblons les briques pour construire un moteur de recherche.
Le principe :

1. L'utilisateur pose une **question en langage naturel**
2. On **encode** cette question avec le même modèle
3. On calcule la **similarité cosinus** entre la question et tous les synopsis
4. On retourne les **Top-K** films les plus similaires

In [7]:
def search_movies(
    query: str,
    movie_list: list[dict],
    movie_embeddings: np.ndarray,
    model: SentenceTransformer,
    top_k: int = 5,
) -> list[dict]:
    """Recherche sémantique de films par similarité cosinus.

    Args:
        query: Requête en langage naturel.
        movie_list: Liste des films (dicts avec title, overview, etc.).
        movie_embeddings: Matrice d'embeddings pré-calculés.
        model: Modèle sentence-transformers.
        top_k: Nombre de résultats à retourner.

    Returns:
        Liste des top_k films les plus similaires avec leur score.
    """
    # 1. Encoder la requête
    query_vec = model.encode([query])

    # 2. Calculer la similarité avec tous les films
    similarities = cosine_similarity(query_vec, movie_embeddings)[0]

    # 3. Trier par score décroissant
    top_indices = np.argsort(similarities)[::-1][:top_k]

    # 4. Construire les résultats
    results = []
    for idx in top_indices:
        results.append({
            "rank": len(results) + 1,
            "title": movie_list[idx]["title"],
            "score": float(similarities[idx]),
            "vote_average": movie_list[idx]["vote_average"],
            "overview": movie_list[idx]["overview"][:120] + "...",
        })
    return results

In [8]:
def display_results(query: str, results: list[dict]) -> None:
    """Affiche les résultats de recherche de manière lisible."""
    print(f'🔍 Requête : "{query}"')
    print(f"{'─' * 60}")
    for r in results:
        print(f"  #{r['rank']}  {r['score']:.4f}  ★{r['vote_average']}  {r['title']}")
        print(f"       {r['overview']}")
        print()


# Test !
results = search_movies("film de zombies effrayant", movies, embeddings, model)
display_results("film de zombies effrayant", results)

🔍 Requête : "film de zombies effrayant"
────────────────────────────────────────────────────────────
  #1  0.5451  ★7.26  Bienvenue à Zombieland
       Dans un monde infesté de zombies, deux hommes tentent de survivre. Columbus, le plus jeune, est terrorisé à l'idée d'êtr...

  #2  0.5253  ★6.82  Blade
       Mi-homme mi-vampire, possédant toutes les qualités des goules et aucune de leur faiblesse, capable de vivre en plein jou...

  #3  0.5244  ★7.425  Scream
       Casey Becker, une belle adolescente, est seule dans la maison familiale. Elle s'apprête à regarder un film d'horreur, ma...

  #4  0.5086  ★7.294  Conjuring 2 : Le Cas Enfield
       Ce thriller paranormal nous présente une nouvelle fois un cas réel tiré des archives des célèbres démonologues Ed et Lor...

  #5  0.4744  ★7.187  Sleepy Hollow, la légende du cavalier sans tête
       En 1799, dans une bourgade de la Nouvelle-Angleterre, plusieurs cadavres sont successivement retrouvés décapités. Les tê...



---
## Partie 6 — À vous de jouer !

### Exercice 6.1 — Testez différentes requêtes

Lancez des recherches avec des requêtes variées et observez les résultats.
Essayez des formulations différentes pour le même besoin.

In [9]:
# Exercice 6.1 — Testez vos propres requêtes
# Modifiez la requête ci-dessous et exécutez la cellule

requetes_a_tester = [
    "un film avec des fantômes dans une maison hantée",
    "horreur psychologique angoissante",
    "film de vampire moderne",
    # Ajoutez vos propres requêtes ici :
]

for q in requetes_a_tester:
    results = search_movies(q, movies, embeddings, model, top_k=3)
    display_results(q, results)
    print("=" * 60)
    print()

🔍 Requête : "un film avec des fantômes dans une maison hantée"
────────────────────────────────────────────────────────────
  #1  0.6456  ★7.294  Conjuring 2 : Le Cas Enfield
       Ce thriller paranormal nous présente une nouvelle fois un cas réel tiré des archives des célèbres démonologues Ed et Lor...

  #2  0.5986  ★6.943  Insidious
       Alors qu'un jeune couple s'installe dans sa nouvelle maison, l'aîné de leurs enfants tombe dans un coma inexpliqué. Peu ...

  #3  0.4737  ★6.644  La Cabane dans les bois
       Cinq étudiants et amis partent passer le week-end dans une cabane perdue au fond des bois. Ils ne savent pas qu'ils font...


🔍 Requête : "horreur psychologique angoissante"
────────────────────────────────────────────────────────────
  #1  0.5519  ★7.343  Split
       Les fractures mentales des personnes présentant un trouble dissociatif de la personnalité ont longtemps fasciné et échap...

  #2  0.4471  ★8.066  The Thing
       Hiver 1982 au cœur de l’Antarctique. Une é

### Exercice 6.2 — Recherche par mots-clés vs sémantique

Comparez la recherche sémantique avec une recherche par mots-clés classique
(type `if mot in texte`). Complétez la fonction ci-dessous :

In [10]:
def search_keyword(query: str, movie_list: list[dict], top_k: int = 5) -> list[dict]:
    """Recherche par mots-clés simple (baseline).

    Args:
        query: Requête de l'utilisateur.
        movie_list: Liste des films.
        top_k: Nombre de résultats.

    Returns:
        Films dont le synopsis contient au moins un mot de la requête.
    """
    # TODO : Implémentez la recherche par mots-clés
    # Indice : découpez la requête en mots, comptez combien de mots
    # apparaissent dans chaque synopsis, triez par ce compte.
    query_words = query.lower().split()
    scored = []
    for movie in movie_list:
        overview_lower = movie["overview"].lower()
        # ── Votre code ici ──
        count = 0  # Remplacez par le bon calcul
        # ────────────────────
        scored.append({"title": movie["title"], "score": count})

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]


# Comparez les deux approches
query = "film de zombies effrayant"
print("=== Recherche SÉMANTIQUE ===")
for r in search_movies(query, movies, embeddings, model, top_k=3):
    print(f"  {r['score']:.4f}  {r['title']}")

print("\n=== Recherche MOTS-CLÉS ===")
for r in search_keyword(query, movies, top_k=3):
    print(f"  {r['score']}  {r['title']}")

=== Recherche SÉMANTIQUE ===
  0.5451  Bienvenue à Zombieland
  0.5253  Blade
  0.5244  Scream

=== Recherche MOTS-CLÉS ===
  0  Ça
  0  Shining
  0  Get Out


<details>
<summary>💡 Solution exercice 6.2</summary>

```python
count = sum(1 for word in query_words if word in overview_lower)
```

Vous devriez constater que la recherche sémantique retrouve des films
pertinents même quand les mots exacts n'apparaissent pas dans le synopsis.
Par exemple, "zombies" trouvera des films parlant de "morts-vivants".
</details>

### Exercice 6.3 — Visualiser les embeddings

Les embeddings ont 384 dimensions — impossible à visualiser directement.
On peut utiliser une réduction de dimensionnalité (comme en IA prédictive !)
pour projeter en 2D :

In [ ]:
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")

# Réduction à 2 dimensions
pca = PCA(n_components=2, random_state=42)
embeddings_2d = pca.fit_transform(embeddings)

print(f"Variance expliquée : {pca.explained_variance_ratio_.sum():.2%}")
print(f"\nProjection 2D des {len(movies)} films :")
print(f"{'Film':<40} {'X':>8} {'Y':>8} {'Note':>5}")
print("─" * 65)
for i, movie in enumerate(movies[:10]):
    title = movie['title'][:38]
    print(f"{title:<40} {embeddings_2d[i][0]:>8.3f} {embeddings_2d[i][1]:>8.3f} {movie['vote_average']:>5.1f}")

print(f"... et {len(movies) - 10} autres films")

> **Note** : Avec seulement 2 dimensions, on perd beaucoup d'information
> (voir la variance expliquée). En production, on travaille toujours
> avec les 384 dimensions complètes. Cette visualisation est uniquement
> pédagogique.

---
## Récapitulatif

Dans cet exercice, vous avez :

✅ **Encodé** des textes en vecteurs avec `sentence-transformers`
✅ **Calculé** la similarité cosinus (manuellement puis avec sklearn)
✅ **Construit** un moteur de recherche sémantique fonctionnel
✅ **Comparé** recherche sémantique vs mots-clés

### Ce qu'on retient

| Concept | En IA prédictive | En IA générative (RAG) |
|---------|-----------------|----------------------|
| Features | Variables numériques | Vecteurs d'embeddings (384d) |
| Distance | Distance euclidienne | Similarité cosinus |
| Recherche | Filtres SQL, WHERE | Recherche vectorielle (pgvector) |

### Prochaine étape

Dans le **notebook 02**, on va apprendre à **classifier l'intention**
de l'utilisateur avant de lancer la recherche. Cela permet de ne pas
chercher dans la base de données quand l'utilisateur dit simplement "Bonjour !" 